# Predicting Auto Insurance Claim *Frequency*: A Poisson Model

A Poisson regression model for how *often* motor insurance policyholders claim,
built on approximately 678,000 French motor policies. This is the frequency half of insurance
pricing. The companion to a severity model (how *much* they claim). Together,
frequency × severity is how insurers actually price risk.

## The question

How often will a given motor policyholder claim in a year, and which risk
factors drive that frequency? Unlike a severity model (claim size), frequency
modelling has to deal with two things that make it distinctly actuarial: **rare
events** (most people never claim) and **exposure** (policies are observed for
different lengths of time).


## The data

- Approx. 678,000 policies (French motor third-party liability, "freMTPL")
- Target: **ClaimNb** (number of claims per policy)
- Key column: **Exposure** (the fraction of a year each policy was observed)
- Risk factors: driver age, vehicle age and power, fuel type, brand, the
  bonus-malus (no-claims) score, region, area, and population density

**The shape of the target matters.** About **95% of policies had zero claims**;
approx. 4.7% had one; the rest a small tail. This is classic low-frequency insurance
data and it means ordinary *accuracy* is a useless metric here: a model that
predicts "zero for everyone" would be 95% accurate and completely worthless.

In [1]:
import pandas as pd
df = pd.read_csv("freMTPL2freq.csv")
print(df.head())
print(df.columns)   
print(df["ClaimNb"].value_counts())
print("Total policies:", len(df))

   IDpol  ClaimNb  Exposure  VehPower  VehAge  DrivAge  BonusMalus VehBrand  \
0    1.0        1      0.10         5       0       55          50      B12   
1    3.0        1      0.77         5       0       55          50      B12   
2    5.0        1      0.75         6       2       52          50      B12   
3   10.0        1      0.09         7       0       46          50      B12   
4   11.0        1      0.84         7       0       46          50      B12   

    VehGas Area  Density       Region  
0  Regular    D     1217  Rhone-Alpes  
1  Regular    D     1217  Rhone-Alpes  
2   Diesel    B       54     Picardie  
3   Diesel    B       76    Aquitaine  
4   Diesel    B       76    Aquitaine  
Index(['IDpol', 'ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge',
       'BonusMalus', 'VehBrand', 'VehGas', 'Area', 'Density', 'Region'],
      dtype='str')
ClaimNb
0     643953
1      32178
2       1784
3         82
4          7
11         3
5          2
6          1
8        

## Method

- **Exposure offset.** A policy observed for a full year has far more opportunity
  to claim than one observed for a month, so raw counts aren't comparable. The
  model predicts a claim *rate* (claims per year of exposure) and weights each
  policy by its exposure, the standard way to handle the offset. This is the
  single most important step that makes it a genuine frequency model rather than
  a naive one.
- **Encoding.** Categorical factors (region, brand, fuel, area) one-hot encoded.
- **Scaling.** Features standardised so the optimiser converges cleanly (the
  columns ranged from single digits to population densities in the thousands).
- **Model.** Poisson regression (the model built for count data).
- **Split.** 80% train (approx. 542k) / 20% test (approx. 136k).
- **Evaluation.** Mean Poisson deviance (the right metric for counts), compared
  against a naive baseline that gives everyone the overall average claim rate.

## Findings

From our findings we found out that:
- **Bonus-malus score** raised claim frequency the most (worse no-claims
  history predicts more claims).
- **Vehicle age** and **driver age** lowered it (older cars and older drivers
  claim less often).
- **Urban/denser areas** raised frequency while **rural areas** lowered it, consistent
  with more traffic meaning more incidents.

Every major factor moves in the direction real insurance experience would
predict, which is a good sign the model learned something genuine rather than
noise.

**But the predictive lift is modest.** The model's Poisson deviance (approx. 0.3293)
was only slightly better than the naive baseline (approx. 0.3314). In other words, the
risk factors *do* help but only a little.


In [2]:
from sklearn.model_selection import train_test_split

# separate target, exposure, and features
y = df["ClaimNb"]
exposure = df["Exposure"]

feature_cols = ["VehPower", "VehAge", "DrivAge", "BonusMalus",
                "VehBrand", "VehGas", "Area", "Density", "Region"]
X = pd.get_dummies(df[feature_cols])

# split everything the same way, keeping exposure aligned with its rows
X_train, X_test, y_train, y_test, exp_train, exp_test = train_test_split(
    X, y, exposure, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor

X_train = X_train.astype(float)
X_test = X_test.astype(float)

# put all features on a comparable scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = PoissonRegressor(max_iter=500)
model.fit(X_train_scaled, y_train / exp_train, sample_weight=exp_train)

print("Model trained cleanly.")
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Effect on Claim Rate": model.coef_
}).sort_values("Effect on Claim Rate", ascending=False)

print(coefficients)
from sklearn.metrics import mean_poisson_deviance

# predicted claim RATE per policy, scaled back up by exposure to get expected counts
pred_rate = model.predict(X_test_scaled)
pred_counts = pred_rate * exp_test

deviance = mean_poisson_deviance(y_test, pred_counts)
print("Mean Poisson deviance (lower = better):", deviance)

# compare to a naive baseline: everyone gets the overall average rate
baseline_rate = (y_train.sum() / exp_train.sum())
baseline_counts = baseline_rate * exp_test
baseline_deviance = mean_poisson_deviance(y_test, baseline_counts)
print("Baseline deviance:", baseline_deviance)

Training rows: 542410
Testing rows: 135603
Model trained cleanly.
                               Feature  Effect on Claim Rate
3                           BonusMalus              0.032450
8                         VehBrand_B12              0.013009
22                              Area_E              0.007687
4                              Density              0.007326
35                Region_Ile-de-France              0.006211
23                              Area_F              0.003761
21                              Area_D              0.003671
44                  Region_Rhone-Alpes              0.003557
17                      VehGas_Regular              0.002381
32                        Region_Corse              0.002055
41                     Region_Picardie              0.001838
43  Region_Provence-Alpes-Cotes-D'Azur              0.001728
31            Region_Champagne-Ardenne              0.001104
0                             VehPower              0.001007
36         Region_L

## Takeaway

This is the important finding: **individual claim occurrence is
dominated by randomness.** Risk factors shift the odds in somewhat sensible directions,
but they don't determine whether a specific driver claims next year, that's
mostly luck. This is the fundamental nature of insurance frequency, and it's
exactly *why* insurers price by pooling thousands of policies rather than
predicting individuals. A model claiming to predict individual claims with high
confidence would be a red flag, not a success.

## What I'd do next

The next thing I'd do will be pairing this model with a severity model to get a full frequency x severity price. I will also try using a model that catch non-linear patterns and comapre its deviance. From there I will clean any implausible records that might distort the results, such as policies with exposure over 1 or with unrealistically high claim counts.